In [ ]:
# Import
import random
import numpy as np
import torch
import json
from tqdm import tqdm
from pathlib import Path
import copy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
import os
import csv
from transformers import RobertaModel, RobertaTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from collections import Counter, deque, defaultdict
import re
from rank_bm25 import BM25Okapi

device = "cuda" if torch.cuda.is_available() else "cpu"

# Seed for reproductibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Default paths
ROOT = Path("../Amazon_products") # Root Amazon_products directory
TRAIN_DIR = ROOT / "train"
TEST_DIR = ROOT / "test"

TEST_CORPUS_PATH = os.path.join(TEST_DIR, "test_corpus.txt")  # product_id \t text
TRAIN_CORPUS_PATH = os.path.join(TRAIN_DIR, "train_corpus.txt")

CLASS_HIERARCHY_PATH = ROOT / "class_hierarchy.txt" 
CLASS_RELATED_PATH = ROOT / "class_related_keywords.txt" 
CLASS_PATH = ROOT / "classes.txt" 

SUBMISSION_PATH = "../Submission/submission.csv"  # output file

# Constants
NUM_CLASSES = 531  # total number of classes (0–530)

# Loading functions
def load_classic(path):
    """Load doc into {id: text} dictionary."""
    id2text = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) == 2:
                id, text = parts
                id2text[id] = text
    return id2text

def load_multilabel(path):
    """Load multi-label data into {id: [labels]} dictionary -> for class_hierarchy"""
    id2labels = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                pid, label = parts
                pid = int(pid)
                label = int(label)
                if pid not in id2labels:
                    id2labels[pid] = []
                id2labels[pid].append(label)
    return id2labels

def load_class_keywords(path):
    """Load class keywords into {class_name: [keywords]} dictionary."""
    class2keywords = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if ":" not in line: # accept only valid format
                continue
            classname, keywords = line.strip().split(":", 1)
            keyword_list = [kw.strip() for kw in keywords.split(",") if kw.strip()]
            class2keywords[classname] = keyword_list
    return class2keywords

# Extraction
id2text_test = load_classic(TEST_CORPUS_PATH) # id -> text test
id_list_test = list(id2text_test.keys()) # list id test
print(list(id2text_test.items())[:1])
print(len(id_list_test)) #19658

id2text_train = load_classic(TRAIN_CORPUS_PATH) # id -> text train
id_list_train = list(id2text_train.keys()) # list id train
print(list(id2text_train.items())[:1])
print(len(id_list_train)) #29487

id2class = load_classic(CLASS_PATH) # id class -> class text
print(list(id2class.items())[:1])
print(len(id2class)) #531

class2hierarchy = load_multilabel(CLASS_HIERARCHY_PATH) # id parents -> children (taxonomy)
print(list(class2hierarchy.items())[:1])
print(len(class2hierarchy)) #69

class2related = load_class_keywords(CLASS_RELATED_PATH) # id class -> related keywords
print(list(class2related.items())[:1])
print(len(class2related)) #531


c:\Users\noamc\Documents\insa_korea\Cours\big data\final proj\project_release\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[('0', "conair cs15tcs professional straight styles straightening iron woah ! sure this straightener looks like all the other crappy straightners in the world , but there 's a twist to this one ! it is my first straightner and i 've had it for about 7 months . i bought it only because i was desperate for a cheap straightener because my hair is very thick , long , wavy ! i 'm looking for a new straighner right now ... but until then this one is doing just fine . if it works for me , it will work for you !")]
19658
[('0', 'omron hem 790it automatic blood pressure monitor with advanced omron health management software so far this machine has worked well and is very simple to use . it is nice to have immediate feedback on the bloodpressure effects of my various exercises , food consumption , and relaxation or stress levels .')]
29487
[('0', 'grocery_gourmet_food')]
531
[(0, [1, 8, 208, 211, 213, 216, 229, 255, 265, 218, 271, 277, 249, 288, 313, 357])]
69
[('grocery_gourmet_food', ['snacks'

In [ ]:
def hierarchy_consistency(silver, hierarchy):
    """Hierarchy consistency in a hierarchy given for our silver labels"""
    ok = 0
    total = 0
    for labels in silver.values():
        L = set(labels)
        for parent, children in hierarchy.items():
            for child in children:
                if child in L:
                    total += 1
                    if parent in L:
                        ok += 1
    return ok / total if total > 0 else 0

def label_coverage(silver_labels, num_classes=531):
    """
    silver_labels : { review_id: [label1, label2, ...] }
    returns coverage_ratio, covered_classes
    """
    covered = set()

    for i, labels in silver_labels.items():
        for lbl in labels:
            if 0 <= lbl < num_classes:
                covered.add(lbl)

    coverage_ratio = len(covered) / num_classes
    return coverage_ratio, sorted(list(covered))

def preprocess_text(text):
    """
    Clean text ONLY for BM25 (NOT for MPNet).
    """
    text = text.lower()
    text = text.replace("_", " ")        
    text = re.sub(r"[>&]", " ", text)
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def bm25_tokenize(text):
    return preprocess_text(text).split()



In [3]:
def expand_with_hierarchy_bm25(label, tree, bm25_scores, threshold=0.7):
    """
    BM25-only hierarchical expansion.
    """

    # Build child -> parents map
    parent_map = {}
    for parent, children in tree.items():
        for child in children:
            parent_map.setdefault(child, []).append(parent)

    path = [label]
    score_label = bm25_scores[label]

    parents = parent_map.get(label, [])
    if not parents:
        return path

    # find best parent
    parent_scores = [(p, bm25_scores[p]) for p in parents]
    parent, score_parent = max(parent_scores, key=lambda x: x[1])

    path.append(parent)

    # add 1 grand parent if threshold respected
    grandparents = parent_map.get(parent, [])
    if not grandparents:
        return path

    gp_scores = [(gp, bm25_scores[gp]) for gp in grandparents]
    grandparent, score_gp = max(gp_scores, key=lambda x: x[1])

    if score_gp >= threshold * score_label:
        path.append(grandparent)

    return path


In [ ]:
def clean_token(text: str) -> str:
    """
    Replace underscores by spaces AND split camel-like tokens cleanly.
    Also strip spaces.
    """
    if not isinstance(text, str):
        return text
    
    # Replace underscores with space
    text = text.replace("_", " ")
    text = " ".join(text.split())
    
    return text


def get_enriched_category_with_hierarchy(class_id, id2class, class2related, class_hierarchy):
    """
    Build an enriched text description for a given class.
    The description includes: the class name (cleaned), the names of its parents,
    and any related keywords linked to the class. This is meant to provide a
    richer textual representation for embedding models.
    """
    class_name = clean_token(id2class[str(class_id)])
    
    # Parents
    parents = class_hierarchy.get(str(class_id), {}).get("parents", [])
    parent_names = [
        clean_token(id2class[str(p)])
        for p in parents if str(p) in id2class
    ]
    
    # Keywords
    keywords = class2related.get(class_name.replace(" ", "_"), [])  # fallback
    keywords = [clean_token(k) for k in keywords]
    
    # Build enriched sentence with natural phrasing
    parts = [f"Category: {class_name}."]
    
    if parent_names:
        parts.append("Parent categories: " + ", ".join(parent_names) + ".")
    
    if keywords:
        parts.append("Related keywords: " + ", ".join(keywords) + ".")
    
    return " ".join(parts)


In [ ]:
def generate_silver_labels(train_texts,train_ids,test_texts,test_ids,id2class,class2related,model, class_hierarchy,output_path_train="SilverRemake/silver_train_fullBM.json"):
    """
    Generate silver labels using ONLY BM25.
    No embeddings, no neural models, no MPNet.
    """

    enriched_categories = [get_enriched_category_with_hierarchy(i, id2class, class2related, class_hierarchy) for i in tqdm(range(NUM_CLASSES), desc="Enriching categories")]

    # Build BM25 over categories
    category_corpus_bm25 = [bm25_tokenize(t) for t in enriched_categories]
    bm25 = BM25Okapi(category_corpus_bm25)

    # Prepare corpus
    all_texts = train_texts + test_texts
    all_texts_bm25 = [preprocess_text(t) for t in all_texts]
    all_ids = train_ids + test_ids

    N_train = len(train_ids)
    N_total = len(all_ids)

    silver_all = {}

    # Assign silver labels
    for i in tqdm(range(N_total), desc="Assigning silver labels (BM25)"):
        pid = all_ids[i]

        # BM25 scoring
        doc_tokens = bm25_tokenize(all_texts_bm25[i])
        bm25_scores = bm25.get_scores(doc_tokens)
        bm25_scores = torch.tensor(bm25_scores, dtype=torch.float32)

        # Top-1 category
        top = torch.argmax(bm25_scores).item()

        # Hierarchical expansion
        expanded = expanded = expand_with_hierarchy_bm25(top, class_hierarchy,bm25_scores, threshold=0.7)

        # Simple fixed scores
        scores_rel = [1.0, 0.7, 0.5]
        scores = {expanded[j]: scores_rel[j] for j in range(len(expanded))}

        final_labels = sorted(expanded, key=lambda c: -scores[c])

        if i < N_train:
            key = pid
        else:
            key = N_train + (i - N_train)

        silver_all[key] = {
            "labels": final_labels,
            "scores": [scores[lbl] for lbl in final_labels],
        }

    with open(output_path_train, "w", encoding="utf-8") as f:
        json.dump(silver_all, f, indent=2, ensure_ascii=False)

    return silver_all


In [ ]:
# Exec
print("Generating silver labels")

silver = generate_silver_labels(
    list(id2text_train.values()),
    id_list_train,
    list(id2text_test.values()),
    id_list_test,
    id2class,
    class2related,
    None,
    class2hierarchy,
    output_path_train="SilverRemake/silver_train_fullBM.json",
)

# Constructions of dict for silvers labels
silver_labels = {
    pid: info["labels"]
    for pid, info in silver.items()
}


# Hierarchical
consistency = hierarchy_consistency(silver_labels, class2hierarchy)
print(f"\nHierarchy Consistency: {consistency:.2%}")
coverage, classes = label_coverage(silver_labels)
print(f"Coverage: {coverage:.2%}")
print(f"Covered classes: {len(classes)}/{NUM_CLASSES}")

Generating silver labels


Assigning silver labels (BM25): 100%|██████████| 49145/49145 [04:06<00:00, 199.30it/s]



Hierarchy Consistency: 52.18%
Coverage: 97.74%
Covered classes: 519/531
